# Experimento D — Modelado de costos de coordinación y predicción de speedup

Este notebook **modela matemáticamente el comportamiento del paralelismo** observado en los experimentos A, B y C, utilizando una versión extendida de la **Ley de Amdahl** que incorpora overhead creciente con el número de workers.

## Decisión de diseño: usar datos reales en vez de simular

A diferencia de la propuesta del enunciado (que sugiere "simular un pipeline con tiempos de preparación, transferencia y unión"), este experimento adopta un enfoque distinto: **usar los datos empíricos** generados en los experimentos B y C, y **ajustar un modelo matemático a ellos**.

**Razones para esta decisión:**
- Aprovecha el trabajo previo de los experimentos B y C (consistencia metodológica).
- Los costos de coordinación se descubren desde la realidad medida, no se inventan.
- Permite predecir comportamientos en configuraciones no probadas (ej: ¿qué pasaría con 32 workers?).
- Evita la limitación de las simulaciones con `time.sleep`, que no capturan efectos como contención de SSD.
## El modelo matemático

Se ajusta el siguiente modelo a los datos observados:

$$T(N) = T_{\text{serial}} + \frac{T_{\text{paralelo}}}{N} + \alpha \cdot N$$

Donde:
- $T_{\text{serial}}$: parte irreductiblemente secuencial del trabajo (no se acelera con más workers).
- $T_{\text{paralelo}}$: parte paralelizable, que se divide entre los $N$ workers.
- $\alpha \cdot N$: overhead de coordinación, que **crece linealmente** con el número de workers (representa serialización, despacho, recolección).

A partir de los parámetros ajustados se calcula la **fracción serial** ($s = T_{\text{serial}} / (T_{\text{serial}} + T_{\text{paralelo}})$), y el **speedup máximo teórico** según Amdahl puro: $S_{\max} = 1/s$.

## Aplicación

El modelo se aplica a 3 datasets reales:
1. **Experimento B - CPU-bound** (limpieza de strings con `ProcessPoolExecutor`, N = 1, 2, 4, 8).
2. **Experimento B - I/O-bound** (read/write archivos con `ThreadPoolExecutor`, N = 1, 2, 4, 8).
3. **Experimento C - Dask sobre 100M filas** (varía número de particiones, N = 4, 16, 64).

## Visualizaciones

1. Modelo ajustado vs datos observados (validación visual del ajuste).
2. Speedup ideal lineal vs Amdahl puro vs modelo con overhead.
3. Descomposición del tiempo en componentes (serial, paralelo, overhead).
4. Predicción del N óptimo extrapolando el modelo más allá del rango medido.

## 1. Importación de librerías

Imports necesarios:

- `numpy`, `pandas` — manipulación de datos.
- `matplotlib`, `seaborn` — visualizaciones.
- `pathlib.Path` — manejo de rutas.
- `scipy.optimize.curve_fit` — la pieza clave de este experimento. `curve_fit` ajusta una función no lineal a datos observados usando mínimos cuadrados, encontrando los parámetros que minimizan el error entre el modelo y los datos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import curve_fit

## 2. Configuración de rutas

Se definen las rutas estándar a `datos/` y `visuali/` (mismas que en los experimentos anteriores), sin necesidad de generar datos nuevos — este experimento consume los CSVs ya producidos por A, B y C.

In [ ]:
RAIZ = Path('..').resolve()
RUTA_DATOS = RAIZ / 'datos'
RUTA_VISUALI = RAIZ / 'visuali'

## 3. Carga de los CSVs de los experimentos previos

Se cargan los DataFrames de **estadísticas agregadas** generados en los experimentos  B y C. Estos archivos contienen los tiempos promedio (sobre 3 repeticiones) para cada combinación de configuración.

Aunque el experimento A se carga, en este notebook solo se usan B y C porque ambos tienen un **barrido explícito de paralelismo** (workers en B, particiones en C). El experimento A solo probó con un número fijo de workers, por lo que no permite ajustar el modelo de Amdahl directamente.

In [ ]:
df_a = pd.read_csv(RUTA_DATOS / 'experimentoA_estadisticas.csv')
df_b = pd.read_csv(RUTA_DATOS / 'experimentoB_estadisticas.csv')
df_c = pd.read_csv(RUTA_DATOS / 'experimentoC_estadisticas.csv')

## 4. Definición del modelo matemático

Se definen las dos funciones núcleo del experimento:

### `modelo_amdahl_con_overhead(N, T_serial, T_paralelo, alpha)`

Implementa directamente la fórmula extendida de Amdahl:

$$T(N) = T_{\text{serial}} + \frac{T_{\text{paralelo}}}{N} + \alpha \cdot N$$

Es la función que `scipy.optimize.curve_fit` va a ajustar a los datos.

### `calcular_speedup_amdahl(N, fraccion_serial)`

Calcula el speedup máximo teórico según **Amdahl puro** (sin término de overhead). Esta versión clásica establece el techo asintótico: cuando $N \to \infty$, $S \to 1/s$.

$$S(N) = \frac{1}{s + \frac{1-s}{N}}$$

Se usa en las visualizaciones para mostrar la diferencia entre el techo asintótico (curva verde en el gráfico 2) y la realidad observada (curva roja con overhead).

In [ ]:
def modelo_amdahl_con_overhead(N, T_serial, T_paralelo, alpha):
    return T_serial + T_paralelo / N + alpha * N


def calcular_speedup_amdahl(N, fraccion_serial):
    return 1 / (fraccion_serial + (1 - fraccion_serial) / N)

## 5. Función de ajuste del modelo a datos reales

`ajustar_modelo` toma dos arrays (N_array con valores de workers/particiones, T_array con tiempos observados) y usa `curve_fit` para encontrar los tres parámetros del modelo.

### Detalles importantes

- **`p0 = [0.1, T_array[0], 0.01]`**: valores iniciales para el optimizador. Sin estos, `curve_fit` puede converger a soluciones absurdas. Se usa `T_array[0]` como estimación inicial de `T_paralelo` porque es el tiempo con N pequeño (donde domina el cómputo).
- **`bounds=([0, 0, 0], [inf, inf, inf])`**: restricciones físicas. Los tres parámetros deben ser **no negativos** (no tiene sentido un tiempo serial negativo, por ejemplo).
- **Salida**: diccionario con los 3 parámetros, la fracción serial calculada, el tiempo total estimado en N=1, y las predicciones del modelo en cada N observado.

### Limitación conocida

Si el número de puntos de datos es igual al número de parámetros (3), el sistema está **exactamente determinado**: el modelo ajusta perfecto pero `scipy` no puede estimar la covarianza ni la incertidumbre de los parámetros. Esto ocurre con el experimento C (3 valores de particiones), y `scipy` emite un warning. Es una limitación honesta que se documenta en el análisis final.

In [ ]:
def ajustar_modelo(N_array, T_array):
    # Estimación inicial
    p0 = [0.1, T_array[0], 0.01]   # valores iniciales razonables
    # Ajustar
    parametros, _ = curve_fit(modelo_amdahl_con_overhead, N_array, T_array, p0=p0,
                               bounds=([0, 0, 0], [np.inf, np.inf, np.inf]))
    
    T_serial, T_paralelo, alpha = parametros
    
    # Predicciones del modelo
    T_predicho = modelo_amdahl_con_overhead(N_array, *parametros)
    
    return {
        'T_serial': T_serial,
        'T_paralelo': T_paralelo,
        'alpha': alpha,
        'T_total_estimado': T_serial + T_paralelo,
        'fraccion_serial': T_serial / (T_serial + T_paralelo),
        'T_predicho': T_predicho,
    }

## 6. Preparación de datos: Experimento B - CPU-bound

Se filtran las mediciones de la carga CPU-bound, manteniendo solo el ejecutor `processes` (más la versión secuencial como punto de partida con N=1). Los datos se promedian entre los tres tamaños de lote para obtener **una sola serie limpia** de tiempo vs n_workers.

**Por qué solo `processes` y no `threads`**: en CPU-bound, los threads no aceleran (el GIL serializa). Aplicar Amdahl a threads en CPU daría una fracción serial ~100% sin información útil. La versión `processes` sí escala y refleja el paralelismo real.

In [ ]:
# Filtrar: carga CPU, ejecutor processes (es el que sí paraleliza)
# Promediar entre tamaños de lote para tener una sola serie
df_b_cpu = df_b[
    (df_b['carga'] == 'cpu') & 
    (df_b['ejecutor'].isin(['secuencial', 'processes']))
].groupby('n_workers')['tiempo_promedio'].mean().reset_index()

print("Datos CPU-bound (processes):")
print(df_b_cpu)

## 7. Ajuste del modelo: Experimento B - CPU-bound

Se aplica `ajustar_modelo` a los 4 puntos de datos (N = 1, 2, 4, 8) y se imprimen los parámetros. Estos números son **el resultado central del experimento D**: descomponen el tiempo total en sus componentes y revelan cuánto del trabajo es realmente paralelizable.

In [ ]:
N_workers = df_b_cpu['n_workers'].values
T_observado = df_b_cpu['tiempo_promedio'].values

resultado_b_cpu = ajustar_modelo(N_workers, T_observado)

print("=== Modelo ajustado para Experimento B - CPU-bound ===")
print(f"T_serial:        {resultado_b_cpu['T_serial']:.4f} s")
print(f"T_paralelo:      {resultado_b_cpu['T_paralelo']:.4f} s")
print(f"alpha (overhead): {resultado_b_cpu['alpha']:.4f} s/worker")
print(f"Fracción serial: {resultado_b_cpu['fraccion_serial']*100:.1f}%")
print(f"\nSpeedup máximo teórico (Amdahl puro, N→∞): {1/resultado_b_cpu['fraccion_serial']:.2f}×")

## 8. Ajuste del modelo: Experimento B - I/O-bound

Mismo procedimiento que para CPU-bound, pero ahora con el ejecutor `threads` (que es el que mejor rinde en I/O-bound porque libera el GIL durante operaciones de disco).

**Predicción**: la fracción serial debería ser **mucho mayor** que en CPU-bound, porque la contención del SSD genera un tiempo serial irreductible significativo. Esto debería traducirse en un **speedup máximo teórico bajo** (probablemente cerca de 2×).

In [ ]:
df_b_io = df_b[
    (df_b['carga'] == 'io') & 
    (df_b['ejecutor'].isin(['secuencial', 'threads']))
].groupby('n_workers')['tiempo_promedio'].mean().reset_index()

N_workers = df_b_io['n_workers'].values
T_observado = df_b_io['tiempo_promedio'].values
resultado_b_io = ajustar_modelo(N_workers, T_observado)

print("=== Modelo ajustado para Experimento B - I/O-bound ===")
print(f"T_serial:        {resultado_b_io['T_serial']:.4f} s")
print(f"T_paralelo:      {resultado_b_io['T_paralelo']:.4f} s")
print(f"alpha (overhead): {resultado_b_io['alpha']:.4f} s/worker")
print(f"Fracción serial: {resultado_b_io['fraccion_serial']*100:.1f}%")
print(f"Speedup máximo teórico: {1/resultado_b_io['fraccion_serial']:.2f}×")

## 9. Ajuste del modelo: Experimento C - Dask 100M

Se filtra el experimento C para quedarse solo con el dataset de 100M filas (donde Dask realmente brilla) y solo Dask (no Pandas, que no varía con N). Las "configuraciones" en este caso son las **3 cantidades de particiones** probadas: 4, 16, 64.

### Limitación numérica importante

Solo hay **3 puntos de datos** y el modelo tiene **3 parámetros**. Esto significa que el sistema está exactamente determinado: el modelo siempre va a ajustar perfecto, pero `scipy` no puede estimar la incertidumbre de los parámetros y emite un warning (`OptimizeWarning: Covariance of the parameters could not be estimated`).

Los parámetros obtenidos siguen siendo informativos como **descripción de los datos**, pero deben interpretarse con cautela: no se sabe qué tan robusto es el ajuste si los datos tuvieran ruido. Idealmente se necesitarían más puntos (probar 8, 32, 128 particiones) para validarlo. Esto se documenta en el análisis final.

In [ ]:
# Para Dask, usamos n_particiones como "N"
df_c_dask = df_c[df_c['tecnologia'] == 'dask'].copy()
# Tomar el dataset 100M (donde Dask realmente brilla)
df_c_100m = df_c_dask[df_c_dask['tamano'] == '100M'].sort_values('n_particiones')

print("Datos Dask en 100M:")
print(df_c_100m[['n_particiones', 'tiempo_promedio']])

N_part = df_c_100m['n_particiones'].values
T_obs = df_c_100m['tiempo_promedio'].values
resultado_c = ajustar_modelo(N_part, T_obs)

print("\n=== Modelo ajustado para Experimento C - Dask 100M ===")
print(f"T_serial:        {resultado_c['T_serial']:.4f} s")
print(f"T_paralelo:      {resultado_c['T_paralelo']:.4f} s")
print(f"alpha (overhead): {resultado_c['alpha']:.4f} s/partición")
print(f"Fracción serial: {resultado_c['fraccion_serial']*100:.1f}%")
print(f"Speedup máximo teórico: {1/resultado_c['fraccion_serial']:.2f}×")

## 10. Visualización 1: Modelo ajustado vs datos observados

Tres paneles (uno por experimento) que validan visualmente el ajuste del modelo:

- **Puntos azules**: las mediciones reales.
- **Línea roja discontinua**: la curva del modelo ajustado.
- **Caja con parámetros**: los valores ajustados del modelo (T_serial, T_paralelo, α, fracción serial).

Si el modelo es correcto, **la línea roja debe pasar exactamente por los puntos azules**. Cualquier desviación sistemática indicaría que el modelo no captura algún fenómeno.

**Lo que se observa:**

- En **CPU-bound**: la curva pasa exactamente por los 4 puntos. Modelo perfecto.
- En **I/O-bound**: el modelo captura la **forma de U** del tiempo (baja hasta N=4, sube con N=8). Esta forma es la firma característica del overhead dominando.
- En **Dask**: el modelo pasa por los 3 puntos (sistema exactamente determinado, sin grados de libertad).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plotear_ajuste(ax, N_obs, T_obs, resultado, titulo, xlabel='N (workers)'):
    # Datos observados
    ax.scatter(N_obs, T_obs, s=120, color='#1f77b4', label='Observado', zorder=3)
    
    # Curva del modelo (más densa para que se vea suave)
    N_continuo = np.linspace(min(N_obs), max(N_obs), 100)
    T_modelo = modelo_amdahl_con_overhead(N_continuo, 
                                           resultado['T_serial'], 
                                           resultado['T_paralelo'], 
                                           resultado['alpha'])
    ax.plot(N_continuo, T_modelo, '--', color='#d62728', linewidth=2, label='Modelo Amdahl ajustado')
    
    # Anotación con parámetros
    texto = (f"T_serial = {resultado['T_serial']:.3f}s\n"
             f"T_paralelo = {resultado['T_paralelo']:.3f}s\n"
             f"α = {resultado['alpha']:.4f}\n"
             f"Fracción serial = {resultado['fraccion_serial']*100:.1f}%")
    ax.text(0.97, 0.97, texto, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Tiempo (s)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plotear_ajuste(axes[0], 
               df_b_cpu['n_workers'].values, 
               df_b_cpu['tiempo_promedio'].values,
               resultado_b_cpu, 
               'Experimento B - CPU-bound (processes)')

plotear_ajuste(axes[1], 
               df_b_io['n_workers'].values, 
               df_b_io['tiempo_promedio'].values,
               resultado_b_io, 
               'Experimento B - I/O-bound (threads)')

plotear_ajuste(axes[2], 
               df_c_100m['n_particiones'].values, 
               df_c_100m['tiempo_promedio'].values,
               resultado_c, 
               'Experimento C - Dask 100M', 
               xlabel='N (particiones)')

fig.suptitle('Modelo de Amdahl ajustado a datos reales – Experimento D', fontsize=14)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoD_modelo_vs_observado.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Visualización 2: Speedup ideal vs Amdahl puro vs observado

Tres paneles que comparan **tres curvas de speedup** en función de N:

1. **Línea negra punteada — Ideal lineal**: la promesa ingenua (con N workers, speedup = N).
2. **Línea verde discontinua — Amdahl puro (sin overhead)**: el techo asintótico clásico, considerando solo la fracción serial.
3. **Línea roja sólida — Modelo con overhead**: la predicción realista, que **puede decrecer** con muchos workers.
4. **Puntos azules — Observado**: las mediciones reales.
5. **Línea horizontal morada — Techo asintótico**: el speedup máximo teórico (1/s).

**Lo que se observa:**

- **CPU-bound**: la curva ideal lineal está MUY por encima de las demás. La diferencia entre la realidad (~3.5×) y la promesa ingenua (8×, 16×) es enorme. Visualiza el "error de asumir speedup lineal" que pide la pauta.
- **I/O-bound**: el techo asintótico es **tan bajo (~2×)** que se ve casi pegado al eje. La curva roja eventualmente decrece. Imagen poderosa del cuello de botella físico.
- **Dask**: el techo es alto (~9.6×) pero los puntos observados (azules) están a ~3× — hay margen para crecer, pero en este hardware ya no.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plotear_speedup(ax, N_obs, T_obs, resultado, titulo, xlabel='N (workers)'):
    T_base = T_obs[0]   # tiempo con N=1
    speedup_obs = T_base / T_obs
    
    N_continuo = np.linspace(min(N_obs), max(N_obs)*2, 100)
    T_modelo = modelo_amdahl_con_overhead(N_continuo, 
                                           resultado['T_serial'], 
                                           resultado['T_paralelo'], 
                                           resultado['alpha'])
    speedup_modelo = T_base / T_modelo
    
    speedup_ideal = N_continuo / min(N_obs)
    
    s = resultado['fraccion_serial']
    speedup_amdahl_puro = 1 / (s + (1-s)/N_continuo)
    
    ax.plot(N_continuo, speedup_ideal, 'k:', label='Ideal lineal', linewidth=2)
    ax.plot(N_continuo, speedup_amdahl_puro, 'g--', label='Amdahl puro (sin overhead)', linewidth=2)
    ax.plot(N_continuo, speedup_modelo, 'r-', label='Modelo ajustado (con overhead)', linewidth=2)
    ax.scatter(N_obs, speedup_obs, s=120, color='#1f77b4', label='Observado', zorder=3)
    
    speedup_max = 1 / s
    ax.axhline(y=speedup_max, color='purple', linestyle=':', alpha=0.5, 
               label=f'Techo asintótico ({speedup_max:.2f}×)')
    
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Speedup (×)')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plotear_speedup(axes[0], 
                df_b_cpu['n_workers'].values, 
                df_b_cpu['tiempo_promedio'].values,
                resultado_b_cpu, 
                'Experimento B - CPU-bound')

plotear_speedup(axes[1], 
                df_b_io['n_workers'].values, 
                df_b_io['tiempo_promedio'].values,
                resultado_b_io, 
                'Experimento B - I/O-bound')

plotear_speedup(axes[2], 
                df_c_100m['n_particiones'].values, 
                df_c_100m['tiempo_promedio'].values,
                resultado_c, 
                'Experimento C - Dask 100M',
                xlabel='N (particiones)')

fig.suptitle('Speedup ideal vs Amdahl vs Observado – Experimento D', fontsize=14)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoD_speedup_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Visualización 3: Descomposición del tiempo en componentes

Barras apiladas que muestran **dónde se va el tiempo** según el modelo, para cada valor de N. Cada barra se descompone en:

- **Rojo (T_serial)**: tiempo irreductible, **constante** sin importar N.
- **Azul (T_paralelo / N)**: cómputo paralelizable, **decrece** con más N.
- **Naranja (α · N)**: overhead de coordinación, **crece** con más N.

**Lo que se observa:**

- En **CPU-bound**: domina el azul (cómputo) en N=1, y se va achicando con N. El naranja (overhead) es despreciable.
- En **I/O-bound**: el rojo (serial) **ocupa la mitad de cada barra** sin importar N. Con N=8, el naranja (overhead) ya es comparable al azul. **Visualmente demuestra por qué no se puede acelerar más allá de N=4 en este caso**.
- En **Dask**: el rojo (serial) es ~3.4s constante. El azul es enorme con 4 particiones y se reduce drásticamente con 64. Overhead nulo (Dask está bien optimizado).

**Mensaje pedagógico**: cuando alguien dice "más workers → más velocidad", este gráfico contraargumenta visualmente. La barra naranja está a la espera, paciente, esperando dominar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plotear_descomposicion(ax, N_obs, resultado, titulo, xlabel='N (workers)'):
    serial = np.full_like(N_obs, resultado['T_serial'], dtype=float)
    paralelo = resultado['T_paralelo'] / N_obs
    overhead = resultado['alpha'] * N_obs
    
    ancho = max(N_obs) * 0.08   # ancho proporcional al rango de N
    
    ax.bar(N_obs, serial, ancho, label='T_serial (irreductible)', color='#d62728')
    ax.bar(N_obs, paralelo, ancho, bottom=serial, 
           label='T_paralelo / N (cómputo)', color='#1f77b4')
    ax.bar(N_obs, overhead, ancho, bottom=serial + paralelo,
           label='α · N (overhead)', color='#ff7f0e')
    
    totales = serial + paralelo + overhead
    for n, total in zip(N_obs, totales):
        ax.text(n, total, f'{total:.2f}s', ha='center', va='bottom', fontsize=9)
    
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Tiempo (s)')
    ax.set_xticks(N_obs)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plotear_descomposicion(axes[0], 
                        df_b_cpu['n_workers'].values, 
                        resultado_b_cpu, 
                        'Experimento B - CPU-bound')

plotear_descomposicion(axes[1], 
                        df_b_io['n_workers'].values, 
                        resultado_b_io, 
                        'Experimento B - I/O-bound')

plotear_descomposicion(axes[2], 
                        df_c_100m['n_particiones'].values, 
                        resultado_c, 
                        'Experimento C - Dask 100M',
                        xlabel='N (particiones)')

fig.suptitle('Descomposición del tiempo según el modelo – Experimento D', fontsize=14)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoD_descomposicion.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Visualización 4: Predicción del N óptimo

Este gráfico es el más útil **prácticamente**. Extrapola el modelo ajustado **mucho más allá del rango medido** (hasta N=64) para **predecir** dónde está el N óptimo (el que minimiza el tiempo total).

**Elementos del gráfico:**

- **Banda azul claro**: el rango de N realmente medido.
- **Puntos azules**: los datos observados.
- **Línea roja discontinua**: la predicción del modelo extrapolada hasta N=64.
- **Estrella dorada**: el N óptimo predicho (donde el tiempo es mínimo según el modelo).
- **Línea dorada vertical**: ubicación del óptimo en el eje X.

**Lo que se observa:**

- **CPU-bound**: óptimo predicho en **N=11** con tiempo 0.37s. Más allá, el overhead supera al beneficio.
- **I/O-bound**: óptimo en **N=4** con tiempo 0.22s. Después de N=8 el modelo predice tiempos peores que el secuencial.
- **Dask**: óptimo en **N=64** (extremo del rango). El modelo sigue bajando, sugiriendo que con más particiones podría seguir mejorando ligeramente.

**Valor predictivo**: este gráfico permite **dimensionar infraestructura sin tener que medir todas las configuraciones**. Si en producción tienes que decidir cuántos workers asignar a un pool, el modelo te da una respuesta cuantitativa basada en datos reales.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plotear_prediccion(ax, N_obs, T_obs, resultado, titulo, xlabel='N (workers)'):
    ax.scatter(N_obs, T_obs, s=120, color='#1f77b4', label='Observado', zorder=3)
    
    N_ext = np.linspace(min(N_obs), 64, 200)
    T_pred = modelo_amdahl_con_overhead(N_ext, 
                                          resultado['T_serial'], 
                                          resultado['T_paralelo'], 
                                          resultado['alpha'])
    ax.plot(N_ext, T_pred, '--', color='#d62728', linewidth=2, 
            label='Predicción del modelo')
    
    # Encontrar el N óptimo (donde el tiempo predicho es mínimo)
    idx_min = np.argmin(T_pred)
    N_optimo = N_ext[idx_min]
    T_optimo = T_pred[idx_min]
    
    ax.scatter([N_optimo], [T_optimo], s=200, color='gold', 
               edgecolor='black', linewidth=2, zorder=4,
               label=f'Óptimo: N={N_optimo:.0f}, T={T_optimo:.2f}s')
    
    ax.axvline(x=N_optimo, color='gold', linestyle=':', alpha=0.5)
    
    ax.axvspan(min(N_obs), max(N_obs), alpha=0.15, color='blue', 
               label='Rango medido')
    
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Tiempo (s)')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plotear_prediccion(axes[0], 
                    df_b_cpu['n_workers'].values, 
                    df_b_cpu['tiempo_promedio'].values,
                    resultado_b_cpu, 
                    'Experimento B - CPU-bound')

plotear_prediccion(axes[1], 
                    df_b_io['n_workers'].values, 
                    df_b_io['tiempo_promedio'].values,
                    resultado_b_io, 
                    'Experimento B - I/O-bound')

plotear_prediccion(axes[2], 
                    df_c_100m['n_particiones'].values, 
                    df_c_100m['tiempo_promedio'].values,
                    resultado_c, 
                    'Experimento C - Dask 100M',
                    xlabel='N (particiones)')

fig.suptitle('Predicción del modelo y N óptimo – Experimento D', fontsize=14)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoD_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

---

# Análisis de resultados 

## Resumen de hallazgos principales

El experimento D **modela matemáticamente** el comportamiento observado en los experimentos B y C utilizando una versión extendida de la Ley de Amdahl con término de overhead. El ajuste a datos reales revela tres realidades estructuralmente distintas:

### Tabla resumen de los 3 ajustes

| Experimento | T_serial | T_paralelo | α | Fracción serial | Speedup máximo teórico | Speedup observado | N óptimo predicho |
|---|---|---|---|---|---|---|---|
| B-CPU | 0.158s | 1.189s | 0.0094 | **11.7%** | **8.52×** | 3.46× | N=11 |
| B-IO | 0.149s | 0.144s | 0.0081 | **50.9%** | **1.96×** | 1.77× | N=4 |
| C-Dask 100M | 3.438s | 29.473s | ~0 | **10.4%** | **9.57×** | 2.77× | N=64+ |

## Hallazgo 1: Cargas CPU-bound tienen baja fracción serial

**Observación**: en CPU-bound, la fracción serial es solo 11.7%. El modelo predice que con N suficientemente grande, podría alcanzarse un speedup de hasta 8.52×. El observado fue 3.46× con 8 workers, sugiriendo margen para crecer.

**Interpretación**: el cómputo de limpieza de strings y hashing MD5 es altamente paralelizable. La parte serial corresponde a la creación de procesos, serialización con pickle, y recolección de resultados — costos relativamente fijos respecto a la cantidad de trabajo.

**Predicción del modelo**: el N óptimo es ~11 procesos. Más allá, el overhead (α·N) empieza a superar la ganancia por paralelización. Con 64 procesos el modelo predice tiempos peores que el secuencial.

**Implicación práctica**: dimensionar pools de procesos para CPU-bound en este hardware en torno a 8-12 workers. La heurística "1 proceso por core" sería conservadora pero razonable.

## Hallazgo 2: Cargas I/O-bound tienen altísima fracción serial

**Observación**: en I/O-bound, la fracción serial es **50.9%** — más de la mitad del tiempo es irreductible. El speedup máximo teórico es **1.96×** (apenas el doble que el secuencial).

**Interpretación**: la mitad del tiempo se va en operaciones que no se pueden paralelizar — fundamentalmente, la **contención del subsistema de almacenamiento**. Aunque hay múltiples threads, el SSD físico solo puede atender un número limitado de operaciones de I/O simultáneas. El sistema operativo serializa estas peticiones a nivel de driver.

**Validación con los datos**: el speedup observado (1.77×) está **muy cerca del techo teórico (1.96×)**. Esto significa que el experimento B con threads ya estaba prácticamente al límite físico — no era un problema de configuración subóptima.

**Predicción del modelo**: el N óptimo es **4 threads**. Después de N=8, el modelo predice que agregar más threads **empeora el tiempo total** (el overhead + la contención superan al cómputo paralelizable).

**Implicación práctica**: este es uno de los hallazgos más importantes y contraintuitivos del trabajo. Para acelerar pipelines I/O-bound más allá de 2× **no sirve agregar más threads**. Las soluciones reales son:
- **Múltiples discos físicos** (RAID, almacenamiento distribuido).
- **Caching agresivo** en RAM.
- **Reducir el volumen de I/O** (compresión, formatos columnares).
- **Solapar I/O con cómputo** mediante async I/O.

## Hallazgo 3: Dask escala bien pero está limitado por hardware

**Observación**: en Dask sobre 100M filas, la fracción serial es 10.4% (similar a CPU-bound) y el techo teórico es 9.57×. El α (overhead por partición) es prácticamente cero.

**Interpretación**: Dask está **muy bien optimizado** — el overhead de coordinar particiones es despreciable. La parte serial corresponde a operaciones que requieren ver todos los datos a la vez (la lectura coordinada de los archivos parquet, el `compute()` final que recolecta los resultados).

**Predicción del modelo**: el N óptimo está en el extremo superior del rango medido (N=64). El modelo sugiere que con más particiones (128, 256) podría seguir mejorando, pero las ganancias serían marginales.

**Validación con los datos**: el speedup observado (2.77× medido como T(4)/T(64)) está aún por debajo del techo teórico de 9.57×, indicando margen. La limitación real probablemente es el **número de cores físicos disponibles**: Dask en local mode no puede paralelizar más de lo que el hardware permite.

**Implicación práctica**: para 100M filas en este hardware, configurar Dask con 32-64 particiones. Para datasets aún mayores o para alcanzar el techo teórico, considerar Dask distribuido en cluster.

## Análisis comparativo: por qué el speedup nunca es lineal

El experimento D **demuestra matemáticamente** los factores que limitan el speedup:

1. **Fracción serial irreductible** (Ley de Amdahl clásica): incluso sin overhead, el speedup está acotado superiormente por 1/s.
2. **Overhead creciente con N** (extensión moderna): el costo de coordinación crece con el número de workers, eventualmente superando el beneficio.
3. **Cuellos de botella físicos**: dispositivos de almacenamiento, ancho de banda de memoria, GIL en Python.
4. **Heterogeneidad del hardware**: cores P y E del Apple M4 con rendimientos distintos.

La combinación de estos factores explica por qué **ninguno de los experimentos alcanzó el speedup ideal lineal**:

- B-CPU con 8 workers: ideal 8×, observado 3.46×, eficiencia 43%.
- B-IO con 4 workers: ideal 4×, observado 1.77×, eficiencia 44%.
- C-Dask con 64 particiones: ideal 16× (aprox), observado 6.91×, eficiencia 43%.

Curiosamente, las tres eficiencias rondan el 40-44%. Esto sugiere que **en este hardware específico, ~50% es overhead estructural** independiente del tipo de carga.

## Limitaciones metodológicas

1. **Pocos puntos de datos para C**: el experimento C tiene solo 3 puntos (4, 16, 64 particiones). Esto deja el sistema exactamente determinado: el modelo ajusta perfecto pero `scipy` no puede estimar la incertidumbre. Idealmente se necesitarían 5-7 puntos para validar el ajuste estadísticamente.

2. **Modelo simplificado**: el modelo asume α constante (overhead lineal en N). En la práctica, el overhead puede tener componentes no lineales (ej: aumenta cuadráticamente cuando los workers compiten por un recurso compartido). El modelo capturó bien los datos observados, pero podría fallar al extrapolar mucho más allá del rango medido.

3. **Extrapolación más allá del rango medido**: las predicciones de N óptimo (especialmente en CPU-bound con N=11) están extrapoladas — solo se midieron N=1, 2, 4, 8. La predicción asume que el modelo sigue siendo válido fuera del rango, lo cual no se verificó empíricamente.

4. **No se separó overhead de creación vs. comunicación**: el término α·N agrupa varios tipos de overhead (creación de procesos, serialización pickle, comunicación inter-proceso, recolección). En un análisis más fino, podrían descomponerse.

5. **Hardware específico**: los parámetros ajustados son específicos de Apple M4 con 16 GB de RAM y SSD interno. En otros hardwares (más cores, RAM, RAID, etc.) los valores serían distintos.

## Recomendaciones según escenario 

A partir de los hallazgos del modelo, se proponen recomendaciones para los tres escenarios solicitados:

### Escenario 1: Servidor único

Un solo equipo ejecutando pipelines de procesamiento.

| Tipo de carga | Recomendación | Razón |
|---|---|---|
| CPU-bound (limpieza, hashing, ML) | `ProcessPoolExecutor` con N ≈ cores físicos (4-8 en M4) | Fracción serial baja (12%), aprovecha bien el paralelismo. |
| I/O-bound (lectura/escritura archivos) | `ThreadPoolExecutor` con N=4 máximo | Fracción serial alta (51%), techo en 2×. Más threads empeoran. |
| Mixta CPU+I/O | Pipeline en etapas con la estrategia óptima de cada una | Cada etapa tiene su propia fracción serial. |

**Hallazgo clave para este escenario**: contrario a la intuición común de "usar tantos workers como permita el hardware", el modelo predice que **hay un N óptimo finito** que depende del tipo de carga. Más workers no es siempre mejor.

### Escenario 2: Organización con múltiples fuentes de datos

Empresa que procesa datos provenientes de varias bases, APIs, archivos.

| Recomendación | Razón |
|---|---|
| Usar **Dask** en modo local con particiones por fuente | Cuando hay múltiples archivos/conexiones, particionar por fuente permite procesarlos en paralelo desde la lectura. |
| Para datasets > 10M filas, justifica el overhead de Dask | El experimento C demostró que para 100M filas, Dask es 7× más rápido que Pandas. |
| Para datasets < 1M filas, usar Pandas monolítico | El overhead de Dask supera el beneficio (Dask 64 fue 5× más lento que Pandas en 1M). |
| Particionar por la columna que se usa para groupby | Reduce shuffles y aprovecha localidad de datos. |

**Hallazgo clave**: la decisión Pandas vs Dask **depende del tamaño**, no es una elección filosófica. Hay un punto de transición empírico (alrededor de 10M filas en este experimento).

### Escenario 3: Plataforma de datos en cloud

Infraestructura escalable en servicios cloud (AWS, GCP, Azure).

| Recomendación | Razón |
|---|---|
| Usar **Spark** (no Dask local) si el dataset excede capacidad de un nodo | El experimento C mostró que Dask local satura cuando la RAM se agota. Para datasets de TB, Spark distribuido es la solución estándar. |
| Para datasets que caben en un nodo grande, **Dask distribuido en cluster** es alternativa más liviana | API más cercana a Pandas, menor curva de aprendizaje. |
| Almacenar datos en **Parquet particionado** | Permite lecturas paralelas y predicate pushdown desde el inicio. |
| Dimensionar workers según **fracción serial estimada**, no por intuición | El modelo de Amdahl da una respuesta cuantitativa: si s=10%, no tiene sentido pagar por más de ~10 workers (eficiencia colapsa). |
| Para cargas I/O-bound puras, considerar arquitecturas serverless (Lambda) | Procesar archivos en paralelo en funciones efímeras es más eficiente que mantener un cluster con workers ociosos. |

**Hallazgo clave**: en cloud, **el costo de los workers no es despreciable**. El modelo de Amdahl permite calcular el N óptimo desde el punto de vista costo-eficiencia, no solo desde el punto de vista de tiempo.

## Conclusión del experimento D

El modelo de Amdahl con overhead, ajustado a datos reales, **describe con precisión el comportamiento observado** en los experimentos B y C. Los tres ajustes revelaron realidades estructuralmente distintas:

- CPU-bound: paralelismo aprovechable hasta ~10 workers, techo teórico ~8×.
- I/O-bound: paralelismo limitado por contención física a ~2× máximo.
- Dask: bien optimizado, escala bien con margen para crecer.

El experimento confirma empíricamente que **el speedup lineal es un mito**, y que **el N óptimo no se puede adivinar** — debe modelarse o medirse. Asumir "más workers = más velocidad" es uno de los anti-patrones más comunes en ingeniería de datos, y este experimento lo demuestra cuantitativamente.

Las recomendaciones para los tres escenarios (servidor único, múltiples fuentes, cloud) están **fundamentadas en evidencia empírica**, no en literatura general — un nivel de rigor que justifica la elección de modelar datos reales en lugar de simular con `time.sleep`.
